PRÉPARATION & NETTOYAGE DES DONNÉES 
- Chargement des données
comprendre la structure globale du jeu de données.

In [20]:
import sys
sys.executable

import pandas as pd
import numpy as np

# Chargement du dataset Vélib
df = pd.read_parquet("data/velib_concat.parquet")

# Dimensions initiales
print("Dimensions initiales :", df.shape)

# Aperçu
df.head()


Dimensions initiales : (6485445, 14)


,ts_utc,tbin_utc,station_id,bikes,capacity,mechanical,ebike,status,lat,lon,name,temp_C,precip_mm,wind_mps
0,2025-11-25 00:00:40.388355,2025-11-25,6245,12,27,11,1,OK,48.866811,2.334388,Ventadour - Opéra,7.2,0.1,3.91
1,2025-11-25 00:00:40.388355,2025-11-25,6293,10,28,5,5,OK,48.867219,2.340463,Mairie du 2ème,7.2,0.1,3.91
2,2025-11-25 00:00:40.388355,2025-11-25,6294,14,23,11,3,OK,48.855258,2.347375,Marché aux fleurs,7.2,0.1,3.91
3,2025-11-25 00:00:40.388355,2025-11-25,6295,18,23,16,2,OK,48.850458,2.352454,Pontoise - La Tournelle,7.2,0.1,3.91
4,2025-11-25 00:00:40.388355,2025-11-25,6296,12,17,5,7,OK,48.857616,2.335831,Institut de France,7.2,0.1,3.91


- Diagnostic de qualité des données
🔍 Valeurs manquantes: Identifier les variables problématiques avant toute transformation.

In [2]:
df.dtypes
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_percentage = (missing_values / len(df)) * 100

quality_report = pd.DataFrame({
    "Valeurs manquantes": missing_values,
    "Pourcentage (%)": missing_percentage.round(2)
})

quality_report[quality_report["Valeurs manquantes"] > 0]

,Valeurs manquantes,Pourcentage (%)
temp_C,1503,0.02
precip_mm,1503,0.02
wind_mps,1503,0.02


-> les variables critiques (bikes, capacity, station_id) n’ont pas de valeurs manquantes


🔍 Détection des doublons

🔍 Détection des incohérences logiques: Ces valeurs sont physiquement impossibles → données erronées.

In [9]:
#diagnostic des doublons
duplicates_count = df.duplicated().sum()
print("doublons :" ,duplicates_count)

# Capacité nulle ou négative
invalid_capacity = df[df["capacity"] <= 0].shape[0]

# Vélos disponibles négatifs
invalid_bikes = df[df["bikes"] < 0].shape[0]

print("Capacités invalides :", invalid_capacity)
print("Vélos disponibles négatifs :", invalid_bikes)

doublons : 0
Capacités invalides : 27208
Vélos disponibles négatifs : 0


Le dataset ne contient aucune ligne dupliquée 
Sur Vélib :
capacité = 0 ou négatives correspond très souvent à :
stations fermées
stations en maintenance
stations désactivées temporairement

Ces lignes ne doivent pas être supprimées aveuglément, car elles reflètent la réalité opérationnelle des stations. Il faut exclure les stations fermées ou non exploitables de l’analyse de disponibilité, seules les stations **ouvertes et exploitables** seront conservées.  
Aucune incohérence détectée pour les vélos disponibles.

- nettoyage des données 

Suppression des doublons: Les doublons faussent les statistiques descriptives et les moyennes.

Traitement des valeurs manquantes

In [15]:
df["status"].value_counts()


status
OK        6325960
CLOSED     159485
Name: count, dtype: Int64

La variable `status` indique l’état de fonctionnement des stations.
Après inspection des modalités, seules les stations avec le statut `OK`
ont été conservées pour l’analyse, correspondant aux stations en service.
Les stations fermées (`CLOSED`) ainsi que celles présentant une capacité
nulle ou négative ont été exclues.

Ainsi, le filtrage final sera :
- stations ouvertes (`status == "OK"`),
- capacité > 0,
- vélos disponibles ≥ 0.


In [16]:
df_clean = df.copy()

# Suppression des doublons
df_clean = df_clean.drop_duplicates()

print("Après suppression des doublons :", df_clean.shape)
#Suppresion des variables critiques manquantes 
critical_columns = [
    "station_id",
    "ts_utc",
    "capacity",
    "bikes",
    "status"
]

df_clean = df_clean.dropna(subset=critical_columns)
print("Après suppression valeurs manquantes critiques :", df_clean.shape)

# Filtrage métier Vélib (stations exploitables)
df_clean = df_clean[df_clean["status"] == "OK"]
df_clean = df_clean[df_clean["capacity"] > 0]
df_clean = df_clean[df_clean["bikes"] >= 0]
print("Après filtrage stations ouvertes et valeurs valides :", df_clean.shape)

#Imputation simple des variables secondaires (météo) 
secondary_columns = ["temp_C", "precip_mm", "wind_mps" 
]

for col in secondary_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())


Après suppression des doublons : (6485445, 14)
Après suppression valeurs manquantes critiques : (6485445, 14)
Après filtrage stations ouvertes et valeurs valides : (6325960, 14)


Le dataset initial comprenait **6 485 445 observations et 14 variables**, incluant des informations sur les stations Vélib, le nombre de vélos disponibles, la capacité des stations, le statut, ainsi que des données météorologiques (température, précipitations, vent).

Pour assurer la qualité des données avant analyse, plusieurs étapes ont été réalisées :

1. **Suppression des doublons**  
   - Toutes les lignes dupliquées ont été supprimées afin d'éviter toute redondance.  
   - Dans ce dataset, **aucun doublon n’a été détecté**, ce qui confirme l’intégrité des enregistrements collectés.

2. **Vérification des valeurs manquantes critiques**  
   - Les colonnes essentielles pour l’analyse de la disponibilité des vélos sont :  
     `station_id`, `ts_utc`, `capacity`, `bikes`, `status`.  
   - Le code a vérifié la présence de valeurs manquantes dans ces colonnes et aurait supprimé les lignes concernées si nécessaire.  
   - **Observation importante** : aucune valeur manquante n’a été détectée dans ces variables critiques.  
     ➔ Cette vérification confirme que les données principales sont complètes et fiables pour l’analyse.

3. **Filtrage des stations exploitables**  
   - Seules les stations **ouvertes** (`status = "OK"`) ont été conservées, excluant les stations fermées ou désactivées temporairement.  
   - Les stations avec **une capacité nulle ou négative** ont également été exclues, car elles ne représentent pas de situations exploitables pour l’analyse de la disponibilité.  
   - Les enregistrements avec un **nombre de vélos négatif** auraient été supprimés, mais aucune anomalie de ce type n’a été détectée dans le dataset.

4. **Imputation des variables secondaires (météo)**  
   - Les colonnes `temp_C`, `precip_mm` et `wind_mps` contiennent très peu de valeurs manquantes (0,02% environ).  
   - Ces valeurs ont été complétées par la **médiane** afin de conserver le maximum d’observations et éviter la perte d’information.

**Résultat final du nettoyage :**  
- **Nombre d’observations :** 6 325 960  
- **Nombre de variables :** 14  



Identifier les valeurs extrêmes avec la méthode IQR

Un outlier est une valeur rare statistiquement, mais pas forcément fausse. ➡️ Ces valeurs sont extrêmes, mais 100 % normales dans Vélib.
bikes = 0 → station vide
bikes = capacity → station pleine

La méthode IQR sert à détecter, pas obligatoirement à supprimer.
Q1 = 25 % des valeurs les plus basses
Q3 = 75 % des valeurs les plus hautes
IQR = Q3 − Q1

borne basse = Q1 − 1.5 × IQR
borne haute = Q3 + 1.5 × IQR

In [22]:

Q1 = df_clean["bikes"].quantile(0.25)
Q3 = df_clean["bikes"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_count = df_clean[
    (df_clean["bikes"] < lower_bound) | (df_clean["bikes"] > upper_bound)
].shape[0]

print("Q1 :", Q1)
print("Q3 :", Q3)
print("IQR :", IQR)
print("Nombre de valeurs considérées comme extrêmes (IQR) :", outliers_count)
print("Shape finale du dataset (aucune suppression) :", df_clean.shape)

Q1 : 4.0
Q3 : 17.0
IQR : 13.0
Nombre de valeurs considérées comme extrêmes (IQR) : 64769
Shape finale du dataset (aucune suppression) : (6130178, 14)


Les valeurs extrêmes ont été identifiées à l’aide de la méthode de l’écart
interquartile (IQR), afin d’évaluer la présence de comportements atypiques
dans la distribution du nombre de vélos disponibles.

Toutefois, dans le contexte du service Vélib’, ces valeurs correspondent
majoritairement à des situations réelles et attendues, telles que des
stations entièrement vides ou complètement saturées lors des heures de
forte affluence. Elles reflètent donc le fonctionnement normal du système
et ne constituent pas des erreurs de mesure.

Par conséquent, aucune suppression automatique des outliers n’a été
effectuée, afin de préserver l’intégrité de l’information et de ne pas
biaiser l’analyse des usages et de la disponibilité des vélos.

- Vérification finale
- Résumé du nettoyage 

In [24]:
# Taille initiale
n_initial = df.shape[0]

# Suppression des doublons
df_step1 = df.drop_duplicates()
n_dedup = df_step1.shape[0]

# Suppression valeurs manquantes critiques
df_step2 = df_step1.dropna(subset=[
    "station_id",
    "ts_utc",
    "capacity",
    "bikes",
    "status"
])
n_na = df_step2.shape[0]

# Filtrage métier Vélib
df_step3 = df_step2[
    (df_step2["status"] == "OK") &
    (df_step2["capacity"] > 0) &
    (df_step2["bikes"] >= 0)
]
n_incoh = df_step3.shape[0]

# Outliers : identifiés mais conservés
df_final = df_step3.copy()
n_outliers = df_final.shape[0]

# Tableau récapitulatif
summary = pd.DataFrame({
    "Étape": [
        "Dataset initial",
        "Après suppression des doublons",
        "Après gestion des valeurs manquantes",
        "Après filtrage des incohérences métier",
        "Après analyse des outliers"
    ],
    "Nombre de lignes": [
        n_initial,
        n_dedup,
        n_na,
        n_incoh,
        n_outliers
    ]
})

summary



,Étape,Nombre de lignes
0,Dataset initial,6485445
1,Après suppression des doublons,6485445
2,Après gestion des valeurs manquantes,6485445
3,Après filtrage des incohérences métier,6325960
4,Après analyse des outliers,6325960


Ce tableau synthétise l’impact global de l’ensemble des opérations de nettoyage
(doublons, valeurs manquantes, incohérences et valeurs aberrantes) sur le volume du jeu de données.
